# Adaptive Traffic Signal Optimizer
## Academic Project: Machine Learning & Priority Queue Traffic Simulation
**Course**: BCA Artificial Intelligence & Data Science

This notebook demonstrates:
1. Loading and cleaning the 30-day historical traffic observation dataset
2. Exploratory Data Analysis (EDA) of hourly patterns, road-specific volume, and queues
3. Chronological feature engineering with temporal and lag variables (preventing data leakage)
4. Training and evaluating a Scikit-learn `RandomForestRegressor` against a naive lag baseline
5. Priority Queue calculation using normalized composite scoring with starvation prevention

In [ ]:
import pandas as pd
import numpy as np
import math
import heapq
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Dataset
df = pd.read_csv('../data/sample_traffic.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Loaded {len(df)} records across roads: {df['road'].unique()}")
df.head()

In [ ]:
# 2. Summary Statistics
road_summary = df.groupby('road').agg({
    'vehicle_count': ['mean', 'max', 'sum'],
    'queue_length': ['mean', 'max'],
    'average_speed': 'mean'
})
road_summary

In [ ]:
# 3. Chronological Feature Engineering
df_feat = df.copy()
df_feat.sort_values(by=['road', 'timestamp'], inplace=True)

df_feat['hour'] = df_feat['timestamp'].dt.hour
df_feat['day_of_week'] = df_feat['timestamp'].dt.weekday
df_feat['is_weekend'] = (df_feat['day_of_week'] >= 5).astype(int)

# Lags per road
df_feat['vehicle_count_lag1'] = df_feat.groupby('road')['vehicle_count'].shift(1)
df_feat['vehicle_count_lag2'] = df_feat.groupby('road')['vehicle_count'].shift(2)
df_feat['queue_length_lag1'] = df_feat.groupby('road')['queue_length'].shift(1)
df_feat['speed_lag1'] = df_feat.groupby('road')['average_speed'].shift(1)

df_feat.dropna(inplace=True)

# One-hot encode road
road_dummies = pd.get_dummies(df_feat['road'], prefix='road', dtype=float)
feature_cols = ['hour', 'day_of_week', 'is_weekend', 'road_North', 'road_South', 'road_East', 'road_West',
                'vehicle_count_lag1', 'vehicle_count_lag2', 'queue_length_lag1', 'speed_lag1']

X = pd.concat([df_feat[['hour', 'day_of_week', 'is_weekend', 'vehicle_count_lag1', 'vehicle_count_lag2', 'queue_length_lag1', 'speed_lag1']], road_dummies], axis=1)[feature_cols]
y = df_feat['vehicle_count']

# Chronological split (80% train, 20% test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# 4. Train RandomForestRegressor
rf = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42)
rf.fit(X_train, y_train)

y_pred = np.clip(rf.predict(X_test), 0, None)
baseline_pred = np.clip(X_test['vehicle_count_lag1'].values, 0, None)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = math.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

b_mae = mean_absolute_error(y_test, baseline_pred)
b_rmse = math.sqrt(mean_squared_error(y_test, baseline_pred))
b_r2 = r2_score(y_test, baseline_pred)

print(f"Random Forest  -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.3f}")
print(f"Lag-1 Baseline -> MAE: {b_mae:.2f}, RMSE: {b_rmse:.2f}, R²: {b_r2:.3f}")

In [ ]:
# 5. Priority Queue Demonstration with Heapq
roads = ["North", "South", "East", "West"]
w1, w2, w3 = 0.40, 0.35, 0.25

demands = {"North": 48.0, "South": 42.0, "East": 22.0, "West": 18.0}
queues = {"North": 14, "South": 10, "East": 4, "West": 2}
wait_times = {"North": 35.0, "South": 30.0, "East": 85.0, "West": 15.0}  # East is starving!

heap = []
for r in roads:
    norm_d = min(1.0, demands[r] / 60.0)
    norm_q = min(1.0, queues[r] / 40.0)
    norm_w = min(1.0, wait_times[r] / 120.0)
    base_score = (w1 * norm_d) + (w2 * norm_q) + (w3 * norm_w)
    
    # Aging safeguard
    aging_bonus = 0.25 if wait_times[r] >= 80.0 else 0.0
    final_score = base_score + aging_bonus
    heapq.heappush(heap, (-final_score, r, final_score, aging_bonus))

print("Priority Queue Schedule Order (Highest Score First):")
while heap:
    _, road, score, aging = heapq.heappop(heap)
    starve_note = "[STARVATION SAFEGUARD TRIGGERED]" if aging > 0 else ""
    print(f" - {road}: Score = {score:.4f} {starve_note}")